#medgemma

In [2]:
! pip install --upgrade --quiet accelerate bitsandbytes transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 33.9 MB/s eta 0:00:00


In [3]:
!pip install faiss-cpu sentence-transformers pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 19.0 MB/s eta 0:00:00


In [4]:
import os
import sys

google_colab = "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT")

if google_colab:
    # Use secret if running in Google Colab
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Store Hugging Face data under `/content` if running in Colab Enterprise
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    # Authenticate with Hugging Face
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

In [5]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import requests
import torch

model_id = "google/medgemma-4b-it"

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [ ]:
# Image attribution: Stillwaterising, CC0, via Wikimedia Commons
image_url = "https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png"
image = Image.open(requests.get(image_url, headers={"User-Agent": "example"}, stream=True).raw)

In [18]:
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "당신은 피부질환 전문가입니다. 아래 문맥을 참고하여 질문에 정확하게 답변하세요. 또한, 더욱 정확한 답변을 위해 이미지 첨부를 권장하여 주세요."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "나 너무 간지러"},
            {"type": "image", "image": image}
        ]
    }
]

NameError: name 'image' is not defined

In [17]:
inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)

NameError: name 'messages' is not defined

In [ ]:
def test_chat(text, image):
  messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "당신은 피부질환 전문가입니다. 아래 문맥을 참고하여 질문에 정확하게 답변하세요. 또한, 더욱 정확한 답변을 위해 이미지 첨부를 권장하여 주세요."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": text},
            {"type": "image", "image": image}
        ]
    }
]

  inputs = processor.apply_chat_template(
      messages, add_generation_prompt=True, tokenize=True,
      return_dict=True, return_tensors="pt"
  ).to(model.device, dtype=torch.bfloat16)

  input_len = inputs["input_ids"].shape[-1]

  with torch.inference_mode():
      generation = model.generate(**inputs, max_new_tokens=200, do_sample=False)
      generation = generation[0][input_len:]

  decoded = processor.decode(generation, skip_special_tokens=True)
  print("input text:", text)
  print("result:", decoded)


In [17]:
path = "/content/drive/MyDrive/Joleop Project/data/train/TestCase_HC/"

In [18]:
test_1 = json.loads(open(os.path.join(path, "TestCase.json"), encoding="utf-8").read())

In [ ]:
test_1

[{'image': 'img/Acne.jpg', 'prompt': '얼굴에 붉고 간지러운게 났어. 이거 뭐니?'},
 {'image': 'img/atopic_dermatitis.jpg',
  'prompt': '손목이 너무 가렵고 아파. 이거 왜 이러는거야?'},
 {'image': 'img/Eczem.jpg', 'prompt': '손에 불긋불긋한게 생겼어. 뭘까?'},
 {'image': 'img/Keratosis.jpg', 'prompt': '목 뒤쪽에 크고 검은점이 두 개 생겼어. 무슨 질병인걸까?'},
 {'image': 'img/Acne.jpg', 'prompt': '얼굴에 붉고 간지러운게 났어. 이거 뭐니?'},
 {'image': 'img/Melanocytic Nevi.jpg',
  'prompt': '점같이 생긴게 생겼는데 단순한 점 맞겠지? 다른 질병일까봐 걱정돼.'},
 {'image': 'img/Melanoma.jpg', 'prompt': '점으로 보이는게 생겼는데 점 맞지? 혹시 다른 질병이니?'},
 {'image': 'img/Normal.jpg', 'prompt': '내 손등 어때. 뭔가 문제가 있니?'},
 {'image': 'img/Seborrheic.jpg', 'prompt': '코에 엄청 큰게 났어. 이거 뭐니?'}]

In [ ]:
for i in range(len(test_1)):
  image_path = path + test_1[i]["image"]
  image = Image.open(image_path)
  test_chat(test_1[i]["prompt"], image)
  print("answer:", test_1[i]["image"])
  print()
  print()

input text: 얼굴에 붉고 간지러운게 났어. 이거 뭐니?
result: 안녕하세요. 피부질환 전문가입니다. 첨부해주신 사진을 통해 다음과 같은 진단을 내릴 수 있습니다.

**가능성 있는 질환:**

*   **포도알 (Folliculitis):** 모낭에 염증이 생기는 질환으로, 붉은 반점과 작은 뾰루지가 나타납니다. 특히 얼굴, 가슴, 등 등에 잘 발생하며, 털이 자라는 부위에서 흔하게 나타납니다.
*   **피부염 (Dermatitis):** 피부의 염증으로, 가려움증, 붉어짐, 발진 등을 동반합니다. 알레르기 반응, 자극 물질, 세균 감염 등으로 인해 발생할 수 있습니다.
*   **여드름 (Acne):** 피지선이 과도하게 분비되어 모공이 막히고 염증이 생기는 질환입니다. 붉은 반점, 
answer: img/Acne.jpg


input text: 손목이 너무 가렵고 아파. 이거 왜 이러는거야?
result: 안녕하세요. 손목이 가려움증과 통증을 동반하고 계시는군요. 사진을 보니 **손목 궤양**으로 보입니다. 

**손목 궤양**은 손목에 발생하는 염증성 피부 질환으로, 주로 손목을 굴리는 동작이나 압박으로 인해 발생합니다. 

**원인:**

*   **손목 굴곡근염 (De Quervain's tenosynovitis):** 손목을 굴곡시키는 근육과 인대가 염증을 일으켜 발생합니다. 주로 손목을 자주 사용하는 직업군이나 운동선수에게 흔하게 발생합니다.
*   **손목 터널 증후군 (Carpal tunnel syndrome):** 손목 안쪽의 신경이 압박되어 발생하는 질환입니다. 손목을 굴곡시키거나 스트레스를 받으면 통증, 저림, 감각 이상 등이 나타날
answer: img/atopic_dermatitis.jpg


input text: 손에 불긋불긋한게 생겼어. 뭘까?
result: 제공해주신 이미지에 따르면 손에 불긋불긋한 병변이 생겼을 가능성이 높은 질환은 다음과 같습니다.

**1. 건선 (Psoriasis):**

*   **특징:*

In [19]:
path_2 = "/content/drive/MyDrive/Joleop Project/data/train/testcase_hm/"
test_2 = json.loads(open(os.path.join(path_2, "testcase.json"), encoding="utf-8").read())
test_2

[{'image': 'img/Melanoma_0.jpg', 'prompt': '볼 한가운데에 이런게 생겼어. 이거 그냥 떼도 되는거야?'},
 {'image': 'img/Acne_21.jpg', 'prompt': '입술 옆에 뭐가 나서 짰는데 노란 진물이 났어. 괜찮은거야?'},
 {'image': 'img/atopic_dermatitis_313.jpg',
  'prompt': '무릎 뒤에 피부가 오돌토돌하게 되고, 너무 가려워 죽겠어. 어떻게 해야해?'},
 {'image': 'img/Eczema_95.jpg',
  'prompt': '팔의 피부가 일어나서 각질이 떨어지고, 너무 가려워 너무 긁어서 피가 좀 나긴 했는데, 심각한 병일까?'},
 {'image': 'img/nevi_15.jpg',
  'prompt': '점 같긴 한데, 일반적인 점이랑은 조금 다르게 생겨서 걱정돼. 혹시 뭔가 문제가 생긴걸까?'},
 {'image': 'img/melanoma_306.jpg',
  'prompt': '점 같긴 한데, 일반적인 점이랑은 조금 다르게 생겨서 걱정돼. 혹시 뭔가 문제가 생긴걸까?'},
 {'image': 'img/Seborrheic_950.jpg',
  'prompt': '검게 된 부분이 딱딱하고, 가끔 가려워 없애고 싶은데, 이떻게 하는게 좋을까'}]

In [ ]:
for i in range(len(test_2)):
  image_path = path_2 + test_2[i]["image"]
  image = Image.open(image_path)
  test_chat(test_2[i]["prompt"], image)
  print("answer:", test_2[i]["image"])
  print()
  print()

input text: 볼 한가운데에 이런게 생겼어. 이거 그냥 떼도 되는거야?
result: 안녕하세요. 피부질환 전문가입니다. 첨부해주신 사진을 통해 볼 중앙에 생긴 이상한 덩어리에 대해 말씀드리겠습니다.

**사진을 통해 보이는 덩어리는 '흑색종'일 가능성이 매우 높습니다.** 흑색종은 피부암의 한 종류로, 피부에 검은색 또는 갈색의 덩어리를 형성하는 특징이 있습니다. 사진 속 덩어리는 다음과 같은 특징을 보입니다.

*   **검은색 또는 갈색:** 흑색종은 일반적으로 검은색 또는 갈색의 덩어리를 형성합니다.
*   **불규칙한 모양:** 흑색종은 규칙적인 모양을 가지지 않고 불규칙한 모양을 보일 수 있습니다.
*   **경계 불분명:** 흑색종의 경계가 명확하지 않고 불분명하게 보일 수 있습니다.
*   **두꺼운 피부:**
answer: img/Melanoma_0.jpg


input text: 입술 옆에 뭐가 나서 짰는데 노란 진물이 났어. 괜찮은거야?
result: 안녕하세요. 입술 옆에 뭐가 나서 짜셨는데 노란 진물이 났다고 하셨습니다. 사진을 보니 입술 주변에 여러 개의 붉은 반점과 덩어리가 보입니다.

**이러한 증상은 다음과 같은 피부 질환의 가능성을 시사합니다.**

*   **포도알 (Folliculitis):** 모낭에 염증이 생기는 질환으로, 붉은 반점과 덩어리가 나타날 수 있습니다.
*   **피부염 (Dermatitis):** 피부가 가려워지고 붉어지는 질환으로, 긁으면 진물이 나고 흉터가 남을 수 있습니다.
*   **악성 흑색종 (Melanoma):** 드물지만, 입술 주변에 악성 흑색종이 발생할 수도 있습니다.

**정확한 진단을 위해서는 피부과 전문의의 진료가 필요
answer: img/Acne_21.jpg


input text: 무릎 뒤에 피부가 오돌토돌하게 되고, 너무 가려워 죽겠어. 어떻게 해야해?
result: 안녕하세요. 무릎 뒤에 피부가 오돌토돌하고 가려운 증상으로 힘드시겠네요. 사진을 보니 **피부염**이 의

# medgemma + Vector Database

In [6]:
import os
import json
import torch
import faiss
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel

In [7]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [8]:
vdb_path = "/content/drive/MyDrive/Joleop Project/data/train/Vector_Database/"

In [9]:
index = faiss.read_index(vdb_path + "faiss_db/clip_skin.index")
metadata = json.load(open(vdb_path + "faiss_db/clip_metadata.json", encoding="utf-8"))

In [10]:
def embed_text(text):
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
    features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy().astype("float32")[0]

def embed_image(image):
    # image가 문자열(경로)이면 열기, 아니면 그대로 사용
    if isinstance(image, str):
        image = Image.open(image).convert("RGB")
    elif not isinstance(image, Image.Image):
        raise ValueError("image는 경로나 PIL.Image.Image 객체여야 합니다.")

    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
    features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy().astype("float32")[0]


In [11]:
def search_vector(vector, k=5):
    D, I = index.search(np.array([vector]), k)
    results = []
    for rank, idx in enumerate(I[0]):
        info = metadata[idx]
        results.append({
            "rank": rank + 1,
            "score": float(D[0][rank]),
            "label": info["label"],
            "modality": info["modality"],
            "content": info["content"]
        })
    return results

In [12]:
def multimodal_search(text_query=None, image_path=None, k=5, alpha=0.5):
    assert text_query or image_path, "❌ text_query 또는 image_path 둘 중 하나는 필요합니다."

    vecs, weights = [], []

    if text_query:
        t_vec = embed_text(text_query)
        vecs.append(t_vec)
        weights.append(alpha)

    if image_path:
        i_vec = embed_image(image_path)
        vecs.append(i_vec)
        weights.append(1 - alpha)

    query_vec = np.average(np.stack(vecs), axis=0, weights=weights).astype("float32")

    results = search_vector(query_vec, k)
    return results

In [13]:
def retrieve_contexts(text=None, image=None, k=3):
    vecs = []
    if text:
        text_vec = embed_text(text)
        vecs.append(text_vec)
    else:
        text_vec = None

    if image:
        image_vec = embed_image(image)
        vecs.append(image_vec)
    else:
        image_vec = None

    if not vecs:
        raise ValueError("텍스트나 이미지는 최소 하나가 필요합니다.")

    query_vec = vecs[0] if len(vecs) == 1 else (text_vec + image_vec) / 2
    query_vec = query_vec / np.linalg.norm(query_vec)
    query_vec = np.expand_dims(query_vec, axis=0).astype("float32")

    D, I = index.search(query_vec, k)
    contexts = [metadata[i]["content"] for i in I[0] if i < len(metadata)]
    return contexts


In [14]:
def medgemma_chat(messages):
    user_text = None
    image = None
    for c in messages[-1]["content"]:
        if c["type"] == "text":
            user_text = c["text"]
        elif c["type"] == "image":
            image = c["image"]

    contexts = retrieve_contexts(user_text, image)
    system_prompt = "당신은 피부질환 전문가입니다. 아래 문맥을 참고하여 질문에 정확하게 답변하세요. 또한, 더욱 정확한 답변을 위해 이미지 첨부를 권장하여 주세요.\n"
    system_prompt += "\n".join(f"- {ctx}" for ctx in contexts)

    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": []}
    ]
    if user_text:
        messages[-1]["content"].append({"type": "text", "text": user_text})
    if image:
        messages[-1]["content"].append({"type": "image", "image": image})

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(**inputs, max_new_tokens=200, do_sample=False)
        generation = generation[0][input_len:]

    decoded = processor.decode(generation, skip_special_tokens=True)
    print(decoded)


In [15]:
def test_rag(text, image):
  messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "당신은 피부질환 전문가입니다. 아래 문맥을 참고하여 질문에 정확하게 답변하세요. 또한, 더욱 정확한 답변을 위해 이미지 첨부를 권장하여 주세요."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": text},
            {"type": "image", "image": image}
        ]
    }
  ]
  print(text)
  medgemma_chat(messages)


In [20]:
for i in range(len(test_1)):
  image_path = path + test_1[i]["image"]
  image = Image.open(image_path)
  test_rag(test_1[i]["prompt"], image)
  print("answer:", test_1[i]["image"])
  print()
  print()

얼굴에 붉고 간지러운게 났어. 이거 뭐니?
제공해주신 사진을 바탕으로 현재 상태를 진단하기는 어렵습니다. 하지만 사진을 통해 몇 가지 가능성을 생각해 볼 수 있습니다.

**가능성 1: 아토피 피부염**

*   **증상:** 붉은 반점, 가려움증, 각질 등 아토피 피부염의 특징적인 증상을 보입니다.
*   **가능성:** 아토피 피부염은 흔한 피부 질환으로, 특히 얼굴, 목, 팔꿈치, 무릎 등에 잘 발생합니다.
*   **추가 정보:** 아토피 피부염은 가려움증이 심하고, 특정 계절이나 환경 변화에 의해 악화될 수 있습니다.

**가능성 2: 알레르기성 피부염**

*   **증상:** 붉은 반점, 가려움증, 두드러기 등 알레르기 반응의 특징적인 증상을 보입니다
answer: img/Acne.jpg


손목이 너무 가렵고 아파. 이거 왜 이러는거야?
제공해주신 이미지와 문맥을 바탕으로 손목 가려움증과 통증의 원인을 추론해 보겠습니다.

**가능성 있는 원인:**

*   **모공성 각화증 (Miliaria):** 손목은 땀샘이 많아 모공성 각화증이 발생하기 쉬운 부위입니다. 특히 여름철 더운 날씨에 땀이 많이 나면 각화증이 악화될 수 있습니다. 각화증은 땀샘이 막혀 땀이 피부 표면으로 나오지 못하고 갇히면서 피부가 붉게 부어오르는 질환입니다.
*   **피부 건조:** 손목은 다른 부위보다 건조하기 쉬우므로 피부가 건조하면 가려움증과 통증을 유발할 수 있습니다.
*   **알레르기 반응:** 특정 물질에 노출되었을 때 알레르
answer: img/atopic_dermatitis.jpg


손에 불긋불긋한게 생겼어. 뭘까?


KeyboardInterrupt: 

In [ ]:
for i in range(len(test_2)):
  image_path = path_2 + test_2[i]["image"]
  image = Image.open(image_path)
  test_chat(test_2[i]["prompt"], image)
  print("answer:", test_2[i]["image"])
  print()
  print()

input text: 볼 한가운데에 이런게 생겼어. 이거 그냥 떼도 되는거야?
result: 안녕하세요. 피부질환 전문가입니다. 첨부해주신 사진을 통해 볼 중앙에 생긴 이상한 덩어리에 대해 말씀드리겠습니다.

**사진을 통해 보이는 덩어리는 '흑색종'일 가능성이 매우 높습니다.** 흑색종은 피부암의 한 종류로, 피부에 검은색 또는 갈색의 덩어리를 형성하는 특징이 있습니다. 사진 속 덩어리는 다음과 같은 특징을 보입니다.

*   **검은색 또는 갈색:** 흑색종은 일반적으로 검은색 또는 갈색의 덩어리를 형성합니다.
*   **불규칙한 모양:** 흑색종은 규칙적인 모양을 가지지 않고 불규칙한 모양을 보일 수 있습니다.
*   **경계 불분명:** 흑색종의 경계가 명확하지 않고 불분명하게 보일 수 있습니다.
*   **두꺼운 피부:**
answer: img/Melanoma_0.jpg


input text: 입술 옆에 뭐가 나서 짰는데 노란 진물이 났어. 괜찮은거야?
result: 안녕하세요. 입술 옆에 뭐가 나서 짜셨는데 노란 진물이 났다고 하셨습니다. 사진을 보니 입술 주변에 여러 개의 붉은 반점과 덩어리가 보입니다.

**이러한 증상은 다음과 같은 피부 질환의 가능성을 시사합니다.**

*   **포도알 (Folliculitis):** 모낭에 염증이 생기는 질환으로, 붉은 반점과 덩어리가 나타날 수 있습니다.
*   **피부염 (Dermatitis):** 피부가 가려워지고 붉어지는 질환으로, 긁으면 진물이 나고 흉터가 남을 수 있습니다.
*   **악성 흑색종 (Melanoma):** 드물지만, 입술 주변에 악성 흑색종이 발생할 수도 있습니다.

**정확한 진단을 위해서는 피부과 전문의의 진료가 필요
answer: img/Acne_21.jpg


input text: 무릎 뒤에 피부가 오돌토돌하게 되고, 너무 가려워 죽겠어. 어떻게 해야해?
result: 안녕하세요. 무릎 뒤에 피부가 오돌토돌하고 가려운 증상으로 힘드시겠네요. 사진을 보니 **피부염**이 의

# medgemma fine-tuned

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 로컬 경로 또는 Hugging Face repo 경로 지정
model_path = "/content/drive/MyDrive/Joleop Project/finetuned_models/medgemma-4b-it-sft-lora-crc100k-skin-disease"  # 예시

# Tokenizer & Model 불러오기
tokenizer = AutoTokenizer.from_pretrained(model_path)
fine_tuned_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",           # GPU 자동 할당
    torch_dtype=torch.bfloat16,  # 필요시 fp16/bf16
).eval()

print("✅ 파인튜닝된 모델 로드 완료")


config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

✅ 파인튜닝된 모델 로드 완료


In [21]:
def test_fine_tuned(text, image):
  messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "당신은 피부질환 전문가입니다. 아래 문맥을 참고하여 질문에 정확하게 답변하세요. 또한, 더욱 정확한 답변을 위해 이미지 첨부를 권장하여 주세요. 문장으로 친절하게 질문에 대한 답변을 이야기해주세요."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": text},
            {"type": "image", "image": image}
        ]
    }
]

  inputs = processor.apply_chat_template(
      messages, add_generation_prompt=True, tokenize=True,
      return_dict=True, return_tensors="pt"
  ).to(fine_tuned_model.device, dtype=torch.bfloat16)

  input_len = inputs["input_ids"].shape[-1]

  with torch.inference_mode():
      generation = fine_tuned_model.generate(**inputs, max_new_tokens=200, do_sample=False)
      generation = generation[0][input_len:]

  decoded = processor.decode(generation, skip_special_tokens=True)
  print("input text:", text)
  print("result:", decoded)


In [22]:
for i in range(len(test_1)):
  image_path = path + test_1[i]["image"]
  image = Image.open(image_path)
  test_fine_tuned(test_1[i]["prompt"], image)
  print("answer:", test_1[i]["image"])
  print()
  print()

input text: 얼굴에 붉고 간지러운게 났어. 이거 뭐니?
result: 이 이미지는 여드름으로 보입니다. 모공이 막혀 피지가 쌓인 모습입니다. 전문의의 진료를 권장드립니다. (여드름가능성이 있습니다.)
answer: img/Acne.jpg


input text: 손목이 너무 가렵고 아파. 이거 왜 이러는거야?
result: 이 이미지는 아토피 피부염으로 보입니다. 가려움이 심하고 피부 장벽이 약해 보입니다. 손으로 자극하지 않는 것이 좋습니다. (아토피 피부염가능성이 있습니다.)
answer: img/atopic_dermatitis.jpg


input text: 손에 불긋불긋한게 생겼어. 뭘까?
result: 이 이미지는 아토피 피부염으로 보입니다. 만성적인 염증 패턴이 관찰됩니다. 손으로 자극하지 않는 것이 좋습니다. (아토피 피부염가능성이 있습니다.)
answer: img/Eczem.jpg


input text: 목 뒤쪽에 크고 검은점이 두 개 생겼어. 무슨 질병인걸까?
result: 이 이미지는 멜라닌 세포모반으로 보입니다. 피부에 어두운 점이 있으며 경계가 뚜렷합니다. 약물치료나 연고 사용을 고려해볼 수 있습니다. (멜라닌 세포모반가능성이 있습니다.)
answer: img/Keratosis.jpg


input text: 얼굴에 붉고 간지러운게 났어. 이거 뭐니?
result: 이 이미지는 여드름으로 보입니다. 모공이 막혀 피지가 쌓인 모습입니다. 전문의의 진료를 권장드립니다. (여드름가능성이 있습니다.)
answer: img/Acne.jpg


input text: 점같이 생긴게 생겼는데 단순한 점 맞겠지? 다른 질병일까봐 걱정돼.
result: 이 이미지는 흑색종으로 보입니다. 점의 색이 불균일하고 경계가 불명확합니다. 전문의의 진료를 권장드립니다. (흑색종가능성이 있습니다.)
answer: img/Melanocytic Nevi.jpg


input text: 점으로 보이는게 생겼는데 점 맞지? 혹시 다른 질병이니?
result: 이 이

In [23]:
for i in range(len(test_2)):
  image_path = path_2 + test_2[i]["image"]
  image = Image.open(image_path)
  test_fine_tuned(test_2[i]["prompt"], image)
  print("answer:", test_2[i]["image"])
  print()
  print()

input text: 볼 한가운데에 이런게 생겼어. 이거 그냥 떼도 되는거야?
result: 이 이미지는 각화증으로 보입니다. 피부 표면이 거칠어 보입니다. 손으로 자극하지 않는 것이 좋습니다. (각화증가능성이 있습니다.)
answer: img/Melanoma_0.jpg


input text: 입술 옆에 뭐가 나서 짰는데 노란 진물이 났어. 괜찮은거야?
result: 이 이미지는 여드름으로 보입니다. 피부에 붉은 뾰루지가 보입니다. 손으로 자극하지 않는 것이 좋습니다. (여드름가능성이 있습니다.)
answer: img/Acne_21.jpg


input text: 무릎 뒤에 피부가 오돌토돌하게 되고, 너무 가려워 죽겠어. 어떻게 해야해?
result: 이 이미지는 아토피 피부염으로 보입니다. 가려움이 심하고 피부 장벽이 약해 보입니다. 전문의의 진료를 권장드립니다. (아토피 피부염으로 추정됩니다.)
answer: img/atopic_dermatitis_313.jpg


input text: 팔의 피부가 일어나서 각질이 떨어지고, 너무 가려워 너무 긁어서 피가 좀 나긴 했는데, 심각한 병일까?
result: 이 이미지는 아토피 피부염으로 보입니다. 가려움이 심하고 피부 장벽이 약해 보입니다. 보습제를 자주 바르면 도움이 됩니다. (아토피 피부염가능성이 있습니다.)
answer: img/Eczema_95.jpg


input text: 점 같긴 한데, 일반적인 점이랑은 조금 다르게 생겨서 걱정돼. 혹시 뭔가 문제가 생긴걸까?
result: 이 이미지는 멜라닌 세포모반으로 보입니다. 피부에 어두운 점이 있으며 경계가 뚜렷합니다. 약물치료나 연고 사용을 고려해볼 수 있습니다. (멜라닌 세포모반가능성이 있습니다.)
answer: img/nevi_15.jpg


input text: 점 같긴 한데, 일반적인 점이랑은 조금 다르게 생겨서 걱정돼. 혹시 뭔가 문제가 생긴걸까?
result: 이 이미지는 흑색종으로 보입니다. 점의 색소가 퍼진 듯한 모양입니다. 전문의의 진료

# medgemma fine-tuned + Vector Database

In [24]:
def  fine_tuned_medgemma_rag(messages):
    user_text = None
    image = None
    for c in messages[-1]["content"]:
        if c["type"] == "text":
            user_text = c["text"]
        elif c["type"] == "image":
            image = c["image"]

    contexts = retrieve_contexts(user_text, image)
    system_prompt = "당신은 피부질환 전문가입니다. 아래 문맥을 참고하여 질문에 정확하게 답변하세요. 또한, 더욱 정확한 답변을 위해 이미지 첨부를 권장하여 주세요.\n"
    system_prompt += "\n".join(f"- {ctx}" for ctx in contexts)

    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": []}
    ]
    if user_text:
        messages[-1]["content"].append({"type": "text", "text": user_text})
    if image:
        messages[-1]["content"].append({"type": "image", "image": image})

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(fine_tuned_model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = fine_tuned_model.generate(**inputs, max_new_tokens=200, do_sample=False)
        generation = generation[0][input_len:]

    decoded = processor.decode(generation, skip_special_tokens=True)
    print(decoded)


In [25]:
def test_fine_tuned_medgemma_rag(text, image):
  messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "당신은 피부질환 전문가입니다. 아래 문맥을 참고하여 질문에 정확하게 답변하세요. 또한, 더욱 정확한 답변을 위해 이미지 첨부를 권장하여 주세요."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": text},
            {"type": "image", "image": image}
        ]
    }
  ]
  print(text)
  fine_tuned_medgemma_rag(messages)


In [26]:
for i in range(len(test_1)):
  image_path = path + test_1[i]["image"]
  image = Image.open(image_path)
  test_fine_tuned_medgemma_rag(test_1[i]["prompt"], image)
  print("answer:", test_1[i]["image"])
  print()
  print()

얼굴에 붉고 간지러운게 났어. 이거 뭐니?
이 이미지는 아토피 피부염으로 보입니다. 피부가 두꺼워지고 붉게 염증이 생겼습니다. 약물치료나 연고 사용을 고려해볼 수 있습니다. (아토피 피부염가능성이 있습니다.)
answer: img/Acne.jpg


손목이 너무 가렵고 아파. 이거 왜 이러는거야?
이 이미지는 아토피 피부염으로 보입니다. 만성적인 염증 패턴이 관찰됩니다. 손으로 자극하지 않는 것이 좋습니다. (아토피 피부염가능성이 있습니다.)
answer: img/atopic_dermatitis.jpg


손에 불긋불긋한게 생겼어. 뭘까?
이 이미지는 아토피 피부염으로 보입니다. 피부가 두꺼워지고 붉게 염증이 생겼습니다. 손으로 자극하지 않는 것이 좋습니다. (아토피 피부염가능성이 있습니다.)
answer: img/Eczem.jpg


목 뒤쪽에 크고 검은점이 두 개 생겼어. 무슨 질병인걸까?
이 이미지는 각화증으로 보입니다. 피부 표면이 거칠어 보입니다. 약물치료나 연고 사용을 고려해볼 수 있습니다. (각화증가능성이 있습니다.)
answer: img/Keratosis.jpg


얼굴에 붉고 간지러운게 났어. 이거 뭐니?
이 이미지는 아토피 피부염으로 보입니다. 피부가 두꺼워지고 붉게 염증이 생겼습니다. 약물치료나 연고 사용을 고려해볼 수 있습니다. (아토피 피부염가능성이 있습니다.)
answer: img/Acne.jpg


점같이 생긴게 생겼는데 단순한 점 맞겠지? 다른 질병일까봐 걱정돼.
이 이미지는 흑색종으로 보입니다. 점의 색이 불균일하고 경계가 불명확합니다. 증상이 지속되면 병원 검진이 필요합니다. (흑색종가능성이 있습니다.)
answer: img/Melanocytic Nevi.jpg


점으로 보이는게 생겼는데 점 맞지? 혹시 다른 질병이니?
이 이미지는 멜라닌 세포모반으로 보입니다. 피부에 어두운 점이 있으며 경계가 뚜렷합니다. 손으로 자극하지 않는 것이 좋습니다. (멜라닌 세포모반가능성이 있습니다.)
answer: img/Melanoma.jpg


In [27]:
for i in range(len(test_2)):
  image_path = path_2 + test_2[i]["image"]
  image = Image.open(image_path)
  test_fine_tuned_medgemma_rag(test_2[i]["prompt"], image)
  print("answer:", test_2[i]["image"])
  print()
  print()

볼 한가운데에 이런게 생겼어. 이거 그냥 떼도 되는거야?
이 이미지는 각화증으로 보입니다. 피부 표면이 거칠어 보입니다. 손으로 자극하지 않는 것이 좋습니다. (각화증가능성이 있습니다.)
answer: img/Melanoma_0.jpg


입술 옆에 뭐가 나서 짰는데 노란 진물이 났어. 괜찮은거야?
이 이미지는 여드름으로 보입니다. 피부에 붉은 뾰루지가 보입니다. 손으로 자극하지 않는 것이 좋습니다. (여드름가능성이 있습니다.)
answer: img/Acne_21.jpg


무릎 뒤에 피부가 오돌토돌하게 되고, 너무 가려워 죽겠어. 어떻게 해야해?
이 이미지는 아토피 피부염으로 보입니다. 피부가 두꺼워지고 붉게 염증이 생겼습니다. 증상이 지속되면 병원 검진이 필요합니다. (아토피 피부염가능성이 있습니다.)
answer: img/atopic_dermatitis_313.jpg


팔의 피부가 일어나서 각질이 떨어지고, 너무 가려워 너무 긁어서 피가 좀 나긴 했는데, 심각한 병일까?
이 이미지는 아토피 피부염으로 보입니다. 가려움이 심하고 피부 장벽이 약해 보입니다. 손으로 자극하지 않는 것이 좋습니다. (아토피 피부염가능성이 있습니다.)
answer: img/Eczema_95.jpg


점 같긴 한데, 일반적인 점이랑은 조금 다르게 생겨서 걱정돼. 혹시 뭔가 문제가 생긴걸까?
이 이미지는 각화증으로 보입니다. 피부 표면이 거칠어 보입니다. 보습제를 자주 바르면 도움이 됩니다. (각화증가능성이 있습니다.)
answer: img/nevi_15.jpg


점 같긴 한데, 일반적인 점이랑은 조금 다르게 생겨서 걱정돼. 혹시 뭔가 문제가 생긴걸까?
이 이미지는 흑색종으로 보입니다. 색소가 퍼진 듯한 모양입니다. 전문의의 진료를 권장드립니다. (흑색종가능성이 있습니다.)
answer: img/melanoma_306.jpg


검게 된 부분이 딱딱하고, 가끔 가려워 없애고 싶은데, 이떻게 하는게 좋을까
이 이미지는 각화증으로 보입니다. 피부 표면이 거칠어 보입니다. 전문의의 

#